# PRAKTEK PEMROGRAMAN SPASIAL — PERTEMUAN 3
## Preprocessing Citra Satelit: Filtering, Cloud Masking, Compositing, dan Ekspor
**Wilayah Kajian** : Sanggala Utara, Kabupaten Tana Toraja, Sulawesi Selatan  
**Nama Mahasiswa** : M. ARIEF ABDILLAH  
**NIM**            : V126241003  
**Dosen Pengampu** : Agus Aris, S.Kel., M.Sc  
**Program Studi**  : Penginderaan Jauh dan Sistem Informasi Geografis — Universitas Hasanuddin (2026)

### 1. Inisialisasi Environment & Library
Mengimpor pustaka `ee` (Google Earth Engine), `geemap` untuk visualisasi peta interaktif, serta pustaka pendukung visualisasi spasial.

In [ ]:
# Install geemap jika dijalankan di Google Colab
# !pip install geemap -q

import ee
import geemap
import datetime

# Inisialisasi Earth Engine
try:
    ee.Initialize()
    print("✅ Earth Engine berhasil diinisialisasi!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize()
    print("✅ Earth Engine berhasil diautentikasi dan diinisialisasi!")

### 2. Memuat Batas AOI Sanggala Utara
Memanggil data poligon batas wilayah Sanggala Utara langsung dari GEE Assets.

In [ ]:
# Panggil aset AOI Sanggala Utara
aoi_fc = ee.FeatureCollection("projects/ma-project-of-indonesian/assets/Sanggala_Utara")
aoi = aoi_fc.geometry()

# Hitung luas wilayah
luas_km2 = aoi.area().divide(1e6).getInfo()
print(f"📍 AOI Sanggala Utara berhasil dimuat.")
print(f"📐 Luas Wilayah : {luas_km2:.2f} km²")

### 3. Fungsi Cloud Masking Landsat 8/9 & Sentinel-2
Mendefinisikan fungsi masking berbasis bit diagnosis `QA_PIXEL` untuk Landsat dan `SCL` untuk Sentinel-2.

In [ ]:
# Fungsi Cloud Masking Landsat 8 & 9 (QA_PIXEL)
def mask_l89sr(image):
    qa = image.select("QA_PIXEL")
    # Bit 4: Cloud, Bit 3: Cloud Shadow, Bit 5: Snow/Ice
    cloud_free  = qa.bitwiseAnd(1 << 4).eq(0)
    shadow_free = qa.bitwiseAnd(1 << 3).eq(0)
    snow_free   = qa.bitwiseAnd(1 << 5).eq(0)
    
    mask = cloud_free.And(shadow_free).And(snow_free)
    # Terapkan mask, normalisasi 0-1, dan bawa metadata
    return image.updateMask(mask).divide(10000).copyProperties(image, image.propertyNames())

# Fungsi Cloud Masking Sentinel-2 (Scene Classification Layer / SCL)
def mask_s2scl(image):
    scl = image.select("SCL")
    # Buang kelas: 0=NoData, 1=Defective, 3=Shadow, 8=CloudMed, 9=CloudHigh, 10=Cirrus
    mask = scl.neq(0).And(scl.neq(1)).And(scl.neq(3)) \
              .And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return image.updateMask(mask).divide(10000).copyProperties(image, image.propertyNames())

### 4. Membangun Composite Multitemporal (T1 & T2) & Indeks NDVI
* **Periode T1 (2015–2016)**: Menggunakan Landsat 8 Surface Reflectance.
* **Periode T2 (2023–2024)**: Menggunakan Landsat 9 Surface Reflectance.
* **Metode**: Median compositing (`.median()`) dan clipping sesuai AOI Sanggala Utara.

In [ ]:
def buat_composite(dataset_id, t_start, t_end, cloud_max, label):
    col = (ee.ImageCollection(dataset_id)
           .filterDate(t_start, t_end)
           .filterBounds(aoi)
           .filterMetadata("CLOUD_COVER", "less_than", cloud_max)
           .map(mask_l89sr))
    
    count = col.size().getInfo()
    print(f"📊 {label} ({t_start} s/d {t_end}) -> Jumlah scene tersaring: {count}")
    return col.median().clip(aoi)

# Bangun Composite T1 dan T2
composite_T1 = buat_composite("LANDSAT/LC08/C02/T1_L2", "2015-01-01", "2016-12-31", 40, "T1 (Landsat 8)")
composite_T2 = buat_composite("LANDSAT/LC09/C02/T1_L2", "2023-01-01", "2024-12-31", 40, "T2 (Landsat 9)")

# Fungsi Tambah Band NDVI: (NIR - Red) / (NIR + Red)
def tambah_ndvi(image):
    ndvi = image.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI")
    return image.addBands(ndvi)

composite_T1_final = tambah_ndvi(composite_T1)
composite_T2_final = tambah_ndvi(composite_T2)
print("✅ Composite T1 dan T2 selesai dibuat dengan penambahan band NDVI!")

### 5. Evaluasi Kualitas Composite (Coverage % & Statistik Spektral)
Mengukur kelengkapan data (apakah ada lubang piksel) dan rata-rata nilai spektral NDVI serta reflektansi permukaan.

In [ ]:
# Hitung Total Piksel Ideal AOI pada resolusi 30m
total_pixel = ee.Number(ee.Image.constant(1).reduceRegion(
    reducer=ee.Reducer.count(), geometry=aoi, scale=30, maxPixels=1e10
).get('constant')).getInfo()

# Hitung Piksel Valid T1 & T2
valid_T1 = ee.Number(composite_T1_final.select("SR_B4").reduceRegion(
    reducer=ee.Reducer.count(), geometry=aoi, scale=30, maxPixels=1e10
).get('SR_B4')).getInfo()

valid_T2 = ee.Number(composite_T2_final.select("SR_B4").reduceRegion(
    reducer=ee.Reducer.count(), geometry=aoi, scale=30, maxPixels=1e10
).get('SR_B4')).getInfo()

cov_T1 = (valid_T1 / total_pixel) * 100
cov_T2 = (valid_T2 / total_pixel) * 100

# Rata-rata Statistik Spektral
stats_T1 = composite_T1_final.select(["SR_B4", "SR_B5", "NDVI"]).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=30, maxPixels=1e10
).getInfo()

stats_T2 = composite_T2_final.select(["SR_B4", "SR_B5", "NDVI"]).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=aoi, scale=30, maxPixels=1e10
).getInfo()

print("=" * 50)
print("HASIL EVALUASI KUALITAS COMPOSITE SANGGALA UTARA")
print("=" * 50)
print(f"Coverage T1 (2015-2016) : {cov_T1:.2f}% (Piksel Valid: {valid_T1:,} / {total_pixel:,})")
print(f"Coverage T2 (2023-2024) : {cov_T2:.2f}% (Piksel Valid: {valid_T2:,} / {total_pixel:,})")
print(f"T1 NDVI Mean           : {stats_T1['NDVI']:.4f}")
print(f"T2 NDVI Mean           : {stats_T2['NDVI']:.4f}")
print(f"T1 Reflektansi B4 Mean : {stats_T1['SR_B4']:.4f}")
print(f"T2 Reflektansi B4 Mean : {stats_T2['SR_B4']:.4f}")

### 6. Perbandingan Sensor Landsat 9 (30m) vs Sentinel-2 (10m) — Tugas C
Menguji kerapatan spasial antara citra Landsat 9 dan Sentinel-2 pada jendela waktu yang sama di tahun 2024.

In [ ]:
# Composite Pembanding 2024
l9_comp = (ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
           .filterDate("2024-01-01", "2024-12-31")
           .filterBounds(aoi)
           .filterMetadata("CLOUD_COVER", "less_than", 40)
           .map(mask_l89sr).median().clip(aoi))

s2_comp = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterDate("2024-01-01", "2024-12-31")
           .filterBounds(aoi)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
           .map(mask_s2scl).median().clip(aoi))

# Hitung jumlah piksel masing-masing sensor
n_l9 = ee.Number(l9_comp.select('SR_B4').reduceRegion(
    reducer=ee.Reducer.count(), geometry=aoi, scale=30, maxPixels=1e10
).get('SR_B4')).getInfo()

n_s2 = ee.Number(s2_comp.select('B4').reduceRegion(
    reducer=ee.Reducer.count(), geometry=aoi, scale=10, maxPixels=1e10
).get('B4')).getInfo()

rasio = round(n_s2 / n_l9)
print("=" * 50)
print("PERBANDINGAN RESOLUSI SENSOR SATELIT (AOI SANGGALA UTARA)")
print("=" * 50)
print(f"Jumlah Piksel Landsat 9 (30m)  : {n_l9:,} piksel")
print(f"Jumlah Piksel Sentinel-2 (10m) : {n_s2:,} piksel")
print(f"Rasio Resolusi (S2 / L9)       : {rasio}x lipat lebih rapat (1 piksel L9 = 9 piksel S2)")

### 7. Visualisasi Peta Interaktif (geemap)
Menampilkan seluruh layer hasil pengolahan: True Color, False Color, NDVI, dan Batas Wilayah AOI Sanggala Utara.

In [ ]:
Map = geemap.Map()
Map.centerObject(aoi, 13)

# Parameter Visualisasi
vis_tc = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0.7, 'max': 1.3, 'gamma': 1.4}
vis_fc = {'bands': ['SR_B5', 'SR_B4', 'SR_B3'], 'min': 0.7, 'max': 2.0}
vis_ndvi = {'bands': ['NDVI'], 'min': -0.2, 'max': 0.8, 'palette': ['red', 'yellow', 'lightgreen', 'darkgreen']}

# Tambahkan Layer
Map.addLayer(composite_T1_final, vis_tc, 'T1 True Color (2015-2016)', shown=False)
Map.addLayer(composite_T1_final, vis_fc, 'T1 False Color (2015-2016)', shown=False)
Map.addLayer(composite_T1_final, vis_ndvi, 'T1 NDVI (2015-2016)', shown=False)

Map.addLayer(composite_T2_final, vis_tc, 'T2 True Color (2023-2024)', shown=True)
Map.addLayer(composite_T2_final, vis_fc, 'T2 False Color (2023-2024)', shown=False)
Map.addLayer(composite_T2_final, vis_ndvi, 'T2 NDVI (2023-2024)', shown=False)

# Batas Administrasi AOI
Map.addLayer(aoi_fc.style(color='red', fillColor='00000000', width=2), {}, 'Batas AOI Sanggala Utara')

# Tampilkan Peta
Map

### 8. Perintah Ekspor Citra Komposit ke Google Drive
Mengekspor berkas GeoTIFF composite T1 dan T2 ke Google Drive pada folder `ProyekGEE_PemSpasial/data_processed`.

In [ ]:
# Task Ekspor T1
task_t1 = ee.batch.Export.image.toDrive(
    image=composite_T1_final,
    description='Composite_T1_Bersih_SanggalaUtara',
    folder='ProyekGEE_PemSpasial/data_processed',
    fileNamePrefix='composite_T1',
    region=aoi,
    scale=30,
    crs='EPSG:4326',
    maxPixels=1e13
)
task_t1.start()
print(f"🚀 Task Ekspor T1 dimulai! ID Task: {task_t1.id}")

# Task Ekspor T2
task_t2 = ee.batch.Export.image.toDrive(
    image=composite_T2_final,
    description='Composite_T2_Bersih_SanggalaUtara',
    folder='ProyekGEE_PemSpasial/data_processed',
    fileNamePrefix='composite_T2',
    region=aoi,
    scale=30,
    crs='EPSG:4326',
    maxPixels=1e13
)
task_t2.start()
print(f"🚀 Task Ekspor T2 dimulai! ID Task: {task_t2.id}")
print("Silakan periksa progress ekspor di https://code.earthengine.google.com/tasks")